# Kaggle: build a vLLM test checkpoint

This is the first half of the end-to-end test. It quantizes TinyLlama with this repository's `Quantizer` and leaves a portable compressed-tensors checkpoint in the notebook output.

Before running it in Kaggle, set **Accelerator → GPU T4 x2** and turn **Internet** on. Do not use the P100 option: the serving half uses vLLM, which needs a newer GPU.

After this notebook succeeds, use **Save Version**, then create a Kaggle Dataset from its `quantized_models` output. Attach that Dataset to `kaggle_serve_checkpoint.ipynb`. Keeping the stages in separate sessions prevents vLLM's Torch installation from contaminating the quantizer environment.

In [ ]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator, then restart the session.'
capability = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)} (compute capability {capability[0]}.{capability[1]})')
print(f'Torch: {torch.__version__}')
assert capability >= (7, 5), 'Choose T4 x2 rather than P100; vLLM requires compute capability 7.5+.'
WORKDIR = Path('/kaggle/working')
REPOSITORY = 'https://github.com/lelouch0204/quantization-profiler.git'
BRANCH = 'kaggle-vllm-test-drive'
CHECKOUT = WORKDIR / 'quantization-profiler'


In [ ]:
if not CHECKOUT.exists():
    !git clone --branch {BRANCH} {REPOSITORY} {CHECKOUT}
else:
    !git -C {CHECKOUT} fetch origin {BRANCH}
    !git -C {CHECKOUT} checkout {BRANCH}
    !git -C {CHECKOUT} pull --ff-only origin {BRANCH}

constraints = WORKDIR / 'quantizer-constraints.txt'
constraints.write_text(f'torch=={torch.__version__}\n')
print(constraints.read_text(), end='')
# Keep Kaggle's working Torch build. Do not add --upgrade-strategy eager here.
%pip install -q -c /kaggle/working/quantizer-constraints.txt 'llmcompressor>=0.13.0' datasets transformers


In [ ]:
# Fail early if the quantizer's recipe imports do not match the installed package set.
from llmcompressor import oneshot
from llmcompressor.modifiers.gptq import GPTQModifier
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform import AWQModifier, SmoothQuantModifier

print('llmcompressor recipe imports succeeded')


In [ ]:
import sys

sys.path.insert(0, str(CHECKOUT / 'src'))
from quantizer import CalibrationConfig, QuantScheme, Quantizer

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_ROOT = WORKDIR / 'quantized_models'
quantizer = Quantizer(
    output_root=OUTPUT_ROOT,
    calibration=CalibrationConfig(num_samples=8, max_seq_length=512),
)
checkpoint = quantizer.quantize(MODEL_ID, QuantScheme.W4A16)
print(f'Checkpoint: {checkpoint}')


In [ ]:
import json

required = ('config.json', 'quant_key.json')
missing = [name for name in required if not (checkpoint / name).is_file()]
assert not missing, f'Missing checkpoint files: {missing}'
print(json.dumps(json.loads((checkpoint / 'quant_key.json').read_text()), indent=2))
print('Files:', len(list(checkpoint.iterdir())))
!du -sh {checkpoint}


## Hand-off

Save a Kaggle Version of this notebook. In its Output area, create a Dataset containing the `quantized_models` directory. The serving notebook will discover the checkpoint automatically after you attach that Dataset as an input.